In [ ]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# Settings
# ============================================================
image_prefix = "r2"   # 改这里：r1, r2, r3, a1a2, a3

image_path = f"{image_prefix}.jpg"
output_csv = f"{image_prefix}_height_results.csv"
output_rotated_image = f"{image_prefix}_height_rotated.png"
output_annotated_image = f"{image_prefix}_height_annotated.png"

known_length_mm = 50.0   # 50 mm = 5 cm


# ============================================================
# Helper functions
# ============================================================
def read_image_bgr(path):
    img = cv2.imread(str(path))
    if img is None:
        raise FileNotFoundError(f"Cannot read image: {path}")
    return img


def resize_for_screen(img, max_width=1400, max_height=900):
    """
    Resize image for screen display, but keep scale factor.
    Coordinates will be converted back to original image.
    """
    h, w = img.shape[:2]
    scale = min(max_width / w, max_height / h, 1.0)
    new_w = int(w * scale)
    new_h = int(h * scale)
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    return resized, scale


def select_single_roi(img, window_name):
    """
    Select one ROI using OpenCV.
    Return x1, y1, x2, y2 in original image coordinates.
    """
    display, scale = resize_for_screen(img)

    roi = cv2.selectROI(window_name, display, showCrosshair=True, fromCenter=False)
    cv2.destroyWindow(window_name)

    x, y, w, h = roi

    if w == 0 or h == 0:
        raise RuntimeError("No rectangle was selected.")

    x1 = x / scale
    y1 = y / scale
    x2 = (x + w) / scale
    y2 = (y + h) / scale

    return x1, y1, x2, y2


def click_two_points(img, window_name):
    """
    Click two points on image using OpenCV.
    Press q when two points are selected.
    Return points in original image coordinates.
    """
    display, scale = resize_for_screen(img)
    shown = display.copy()
    points = []

    def mouse_callback(event, x, y, flags, param):
        nonlocal shown, points

        if event == cv2.EVENT_LBUTTONDOWN:
            if len(points) < 2:
                points.append((x / scale, y / scale))
                cv2.circle(shown, (x, y), 6, (0, 0, 255), -1)

                if len(points) == 2:
                    p1 = tuple(np.round(np.array(points[0]) * scale).astype(int))
                    p2 = tuple(np.round(np.array(points[1]) * scale).astype(int))
                    cv2.line(shown, p1, p2, (0, 0, 255), 2)

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.setMouseCallback(window_name, mouse_callback)

    while True:
        cv2.imshow(window_name, shown)
        key = cv2.waitKey(20) & 0xFF

        if key == ord("q") and len(points) == 2:
            break

        if key == 27:
            cv2.destroyWindow(window_name)
            raise RuntimeError("Point selection cancelled.")

    cv2.destroyWindow(window_name)

    return np.array(points[0]), np.array(points[1])


def rotate_image_keep_all(img, angle_deg):
    """
    Rotate image without cropping.
    Positive angle means counter-clockwise.
    """
    h, w = img.shape[:2]
    center = (w / 2, h / 2)

    M = cv2.getRotationMatrix2D(center, angle_deg, 1.0)

    cos = abs(M[0, 0])
    sin = abs(M[0, 1])

    new_w = int((h * sin) + (w * cos))
    new_h = int((h * cos) + (w * sin))

    M[0, 2] += new_w / 2 - center[0]
    M[1, 2] += new_h / 2 - center[1]

    rotated = cv2.warpAffine(
        img,
        M,
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255),
    )

    return rotated


def select_many_rois(img):
    """
    Select many sample boxes.
    For each sample:
      drag box -> press Enter/Space
    When finished:
      press Esc or Cancel
    """
    rectangles = []
    sample_index = 1

    while True:
        temp = img.copy()

        for i, (x1, y1, x2, y2) in enumerate(rectangles, start=1):
            cv2.rectangle(
                temp,
                (int(x1), int(y1)),
                (int(x2), int(y2)),
                (255, 0, 0),
                3,
            )
            cv2.putText(
                temp,
                str(i),
                (int(x1), max(int(y1) - 8, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 0, 0),
                2,
                cv2.LINE_AA,
            )

        print(f"Draw box for sample {sample_index}. Press Enter/Space to confirm. Press Esc when finished.")

        try:
            x1, y1, x2, y2 = select_single_roi(
                temp,
                f"Sample {sample_index}: draw box, Enter confirm, Esc finish"
            )
        except RuntimeError:
            print("Finished selecting samples.")
            break

        rectangles.append((x1, y1, x2, y2))
        sample_index += 1

    return rectangles


# ============================================================
# Step 1. Load image
# ============================================================
img = read_image_bgr(image_path)


# ============================================================
# Step 2. Select 50 mm on ruler
# ============================================================
print("Step 2: On the ruler, draw a box covering exactly 50 mm.")
print("Because the ruler is vertical, the box height will be used as 50 mm.")
print("Drag box, then press Enter or Space.")

x1, y1, x2, y2 = select_single_roi(
    img,
    "Calibration: draw 50 mm on ruler, Enter confirm"
)

pixel_per_known_length = abs(y2 - y1)
mm_per_pixel = known_length_mm / pixel_per_known_length

print(f"Pixel height for {known_length_mm:.1f} mm: {pixel_per_known_length:.2f} px")
print(f"Calibration factor: {mm_per_pixel:.6f} mm/px")


# ============================================================
# Step 3. Rotation correction
# ============================================================
print("Step 3: Click two points along the top edge of the first row.")
print("After clicking two points, press q.")

p1, p2 = click_two_points(
    img,
    "Rotation correction: click two points, then press q"
)

dx = p2[0] - p1[0]
dy = p2[1] - p1[1]

angle_deg = np.degrees(np.arctan2(dy, dx))
rotated_img = rotate_image_keep_all(img, angle_deg)

print(f"Detected tilt angle: {angle_deg:.3f} degrees")
print(f"Image rotated by: {angle_deg:.3f} degrees")

cv2.imwrite(output_rotated_image, rotated_img)
print(f"Saved rotated image for checking: {output_rotated_image}")


# ============================================================
# Step 4. Manually draw boxes for each sample
# ============================================================
print("Step 4: Draw one box per sample.")
print("For each box, press Enter/Space to save.")
print("When finished, press Esc.")

rectangles = select_many_rois(rotated_img)

print(f"Total samples selected: {len(rectangles)}")


# ============================================================
# Step 5. Calculate heights and export results
# ============================================================
records = []
annotated = rotated_img.copy()

for i, (x1, y1, x2, y2) in enumerate(rectangles, start=1):
    pixel_height = abs(y2 - y1)
    real_height_mm = pixel_height * mm_per_pixel

    records.append({
        "sample_index": i,
        "x1_px": x1,
        "y1_px": y1,
        "x2_px": x2,
        "y2_px": y2,
        "pixel_height": pixel_height,
        "height_mm": real_height_mm,
    })

    cv2.rectangle(
        annotated,
        (int(round(x1)), int(round(y1))),
        (int(round(x2)), int(round(y2))),
        (255, 0, 0),
        3,
    )

    cv2.putText(
        annotated,
        f"{i}: {real_height_mm:.1f} mm",
        (int(round(x1)), max(int(round(y1)) - 8, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 0, 0),
        2,
        cv2.LINE_AA,
    )

df = pd.DataFrame(records)

df.to_csv(output_csv, index=False)
cv2.imwrite(output_annotated_image, annotated)

print("Done.")
print(df)
print(f"Saved annotated image: {output_annotated_image}")

Step 2: On the ruler, draw a box covering exactly 50 mm.
Because the ruler is vertical, the box height will be used as 50 mm.
Drag box, then press Enter or Space.
